In [1]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import pickle
import numpy as np
import os

In [2]:
file  = open('data.txt', 'r', encoding = "utf8")

In [3]:
lines=[]
for i in file:
    lines.append(i)

In [4]:
data=""
for i in lines:
    data = ' '.join(lines)

In [5]:
 data = data.replace('\n', '').replace('\r', '').replace('\ufeff', '').replace('“','').replace('”','')

In [6]:
data = data.split()
data = ' '.join(data)

In [7]:
data[:1000]

"THE ADVENTURES OF SHERLOCK HOLMES Arthur Conan Doyle Table of contents A Scandal in Bohemia The Red-Headed League A Case of Identity The Boscombe Valley Mystery The Five Orange Pips The Man with the Twisted Lip The Adventure of the Blue Carbuncle The Adventure of the Speckled Band The Adventure of the Engineer's Thumb The Adventure of the Noble Bachelor The Adventure of the Beryl Coronet The Adventure of the Copper Beeches A SCANDAL IN BOHEMIA Table of contents Chapter 1 Chapter 2 Chapter 3 CHAPTER I To Sherlock Holmes she is always the woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for Irene Adler. All emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false positio

In [8]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

In [9]:
pickle.dump(tokenizer, open('token.pkl', 'wb'))

In [10]:
sequence_data = tokenizer.texts_to_sequences([data])[0]

In [11]:
sequence_data[:10]

[1, 1561, 5, 129, 34, 647, 4498, 4499, 226, 5]

In [12]:
len(sequence_data)

105879

In [14]:
vocab_size = len(tokenizer.word_index)+1

In [15]:
vocab_size

8200

In [16]:
sequence=[]
for i in range(3, len(sequence_data)):
    words = sequence_data[i-3:i+1]
    sequence.append(words)

In [17]:
len(sequence)

105876

In [18]:
sequence = np.array(sequence)

In [19]:
sequence

array([[   1, 1561,    5,  129],
       [1561,    5,  129,   34],
       [   5,  129,   34,  647],
       ...,
       [  28,    1, 8198, 8199],
       [   1, 8198, 8199, 3187],
       [8198, 8199, 3187, 3186]])

In [20]:
X=[]
y=[]

for i in sequence:
    X.append(i[0:3])
    y.append(i[3])

In [21]:
X=np.array(X)
y=np.array(y)

In [22]:
y = to_categorical(y, num_classes=vocab_size)

In [23]:
y[:5]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

**Building the LSTM Model**

In [24]:
model = Sequential()
model.add(Embedding(vocab_size, 10, input_length = 3))
model.add(LSTM(1000, return_sequences = True))
model.add(LSTM(1000))
model.add(Dense(1000, activation='relu'))
model.add(Dense(vocab_size, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint = ModelCheckpoint("next_words.h5", monitor="loss", verbose = 1, save_best_only=True)
model.compile(loss="categorical_crossentropy", optimizer = Adam(learning_rate = 0.001))
model.fit(X, y, epochs=50, batch_size=64, callbacks=[checkpoint])

Epoch 1/50
1654/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 5.5766
Epoch 1: loss improved from inf to 5.60821, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 42s 24ms/step - loss: 5.5767
Epoch 2/50
1654/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 5.3229
Epoch 2: loss improved from 5.60821 to 5.31310, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 39s 23ms/step - loss: 5.3229
Epoch 3/50
1653/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 5.0649
Epoch 3: loss improved from 5.31310 to 5.07205, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 5.0649
Epoch 4/50
1653/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 4.8658
Epoch 4: loss improved from 5.07205 to 4.87392, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 4.8658
Epoch 5/50
1653/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 4.6620
Epoch 5: loss improved from 4.87392 to 4.69388, saving model to next_words.h5


...

1655/1655 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 0.4324
Epoch 47/50
1655/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4288
Epoch 47: loss improved from 0.48125 to 0.47371, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 0.4288
Epoch 48/50
1655/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4205
Epoch 48: loss improved from 0.47371 to 0.46904, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 0.4205
Epoch 49/50
1654/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4129
Epoch 49: loss improved from 0.46904 to 0.46283, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - loss: 0.4130
Epoch 50/50
1655/1655 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.4098
Epoch 50: loss improved from 0.46283 to 0.45736, saving model to next_words.h5


1655/1655 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - loss: 0.4098


In [41]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

# load model
model = load_model("/content/next_words.h5")

# recreate tokenizer
text = open("/content/data.txt").read()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

max_sequence_len = model.input_shape[1]

def predict_word(model, tokenizer, text):

    sequence = tokenizer.texts_to_sequences([text])[0]

    sequence = pad_sequences([sequence], maxlen=max_sequence_len)

    preds = model.predict(sequence, verbose=0)

    predicted_index = np.argmax(preds)

    predicted_word = ""

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            predicted_word = word
            break

    return predicted_word


while True:

    text = input("Enter your line: ")

    if text == "1":
        break

    words = text.split()
    words = words[-3:]
    text = " ".join(words)

    next_word = predict_word(model, tokenizer, text)

    print("Next word prediction:", next_word)

Enter your line: A Scandal
Next word prediction: in
Enter your line: The Boscombe Valley
Next word prediction: mystery
Enter your line: The Boscombe Valley
Next word prediction: mystery
Enter your line: I had seen little of Holmes lately.
Next word prediction: my
Enter your line: 1


In [42]:
import pickle

# save tokenizer
with open("token.pkl", "wb") as f:
    pickle.dump(tokenizer, f)